# 05 Instrumental Variables

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/blob/main/06-Econometrics/05_Instrumental_Variables.ipynb) [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/main?filepath=06-Econometrics/05_Instrumental_Variables.ipynb) [![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](../LICENSE) [![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)


In [ ]:
# === Environment Setup ===
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from IPython.display import Markdown, display
from linearmodels.iv import IV2SLS

# --- Configuration ---
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 14, 'figure.figsize': (12, 8), 'figure.dpi': 150})
%config InlineBackend.figure_format = 'retina'
np.set_printoptions(suppress=True, linewidth=120, precision=4)

# --- Utility Functions ---


In [ ]:
# --- Visualizing the Exclusion Restriction (DAG) ---
import networkx as nx

fig, ax = plt.subplots(figsize=(8, 6))
G = nx.DiGraph()

# Nodes
G.add_node("Z", pos=(0, 1), label="Instrument (Z)")
G.add_node("X", pos=(1, 0.5), label="Treatment (X)")
G.add_node("Y", pos=(2, 1), label="Outcome (Y)")
G.add_node("U", pos=(1.5, 2), label="Unobserved (U)")

# Edges
G.add_edge("Z", "X", label="Relevance")
G.add_edge("X", "Y", label="Causal Effect")
G.add_edge("U", "Y")
G.add_edge("U", "X", label="Confounding")

# Layout
pos = nx.get_node_attributes(G, 'pos')
labels = nx.get_node_attributes(G, 'label')

# Draw
nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=3000, font_size=10, ax=ax)
nx.draw_networkx_edge_labels(G, pos, edge_labels={("Z","X"): "Relevance", ("U","X"): "Confounding"}, font_size=8)
plt.title("Instrumental Variable DAG")
plt.axis('off')

plt.show()


## Table of Contents

1. [Introduction](#Introduction)


## The Lens: Breaking the Endogeneity Loop
**What problem are we solving?**
When $X$ is correlated with the error term $\epsilon$ (endogeneity), OLS is biased and inconsistent.
*   **Omitted Variable Bias:** Ability affects both education and wages.
*   **Reverse Causality:** Growth affects investment, and investment affects growth.

**Instrumental Variables (IV)** provide a way to cut this link and isolate the causal variation in $X$.

**Why this method?**
We need a variable $Z$ (the instrument) that is correlated with $X$ (relevance) but uncorrelated with $\epsilon$ (exclusion restriction). IV/2SLS uses $Z$ to extract only the "clean" variation in $X$, giving us a consistent estimate of the causal effect.



**Economic question.** In *05 Instrumental Variables*, what must remain economically invariant when the computational representation changes? The computational task only has economic meaning after the estimand and identifying assumptions are explicit. Ask what variation identifies the parameter, which observations act as the comparison group, and what data-generating process would make the estimator fail. A good empirical workflow pairs the point estimate with diagnostics, uncertainty, and at least one falsification or sensitivity check so that precision is not confused with identification.

### Learning Objectives
* **Identify** endogeneity from omitted variables, measurement error, and simultaneity.
* **State** the relevance and exclusion restrictions and assess instrument validity.
* **Estimate** IV models using Two-Stage Least Squares (2SLS) and interpret the Local Average Treatment Effect (LATE).
* **Diagnose** weak instruments using the first-stage F-statistic and conduct over-identification tests.

### Prerequisites
* **OLS:** Linear regression, omitted variable bias, and hypothesis testing (Module 06 - OLS).
* **Causal Inference:** Potential outcomes framework and selection bias (Module 06 - Causal Inference).
* **Python:** `statsmodels` and `linearmodels` for IV estimation.
* **Learning-path prerequisite:** [`04_GMM.ipynb`](04_GMM.ipynb)


> **Learning path:** Building on [`04_GMM.ipynb`](04_GMM.ipynb); next continue with [`06_Regression_Discontinuity.ipynb`](06_Regression_Discontinuity.ipynb).


### 1. The IV Estimator: Two-Stage Least Squares (2SLS)
The most common IV estimator is **Two-Stage Least Squares (2SLS)**. It can be understood intuitively as a two-step procedure:

1.  **First Stage:** We purge the endogenous variable $D$ of its correlation with the error term. We do this by regressing $D$ on the instrument $Z$ and any exogenous controls $X$. This gives us the predicted values, $\hat{D}$. These predicted values represent the part of the variation in $D$ that is explained *only* by the exogenous variables.
    $$ D = \pi_0 + \pi_1 Z + \pi_2 X + v $$
2.  **Second Stage:** We run the original regression, but replace the endogenous variable $D$ with its predicted value from the first stage, $\hat{D}$.
    $$ Y = \beta_0 + \beta_1 \hat{D} + \beta_2 X + u $$
Because $\hat{D}$ is, by construction, a linear combination of the exogenous variables, it is uncorrelated with the error term $u$, and this second-stage regression yields a consistent estimate of the causal effect $\beta_1$.

**Important Note:** While this two-stage procedure is intuitive, one should **never run it manually**. The standard errors from the second-stage OLS are incorrect because they fail to account for the uncertainty in estimating the first stage. Always use specialized software (like `linearmodels` or `Stata`) that computes the correct 2SLS variance-covariance matrix.


### 2. Heterogeneous Effects and the LATE Framework
A crucial insight from Imbens and Angrist (1994) is that when the treatment effect is heterogeneous, IV does not recover the Average Treatment Effect (ATE). Instead, it recovers the **Local Average Treatment Effect (LATE)**.

We can divide the population into four groups based on their potential response to a binary instrument $Z$:
1.  **Compliers:** People who take the treatment if encouraged ($D(1)=1$) but not if unencouraged ($D(0)=0$). These are the people whose behavior is changed by the instrument.
2.  **Always-Takers:** People who always take the treatment, regardless of the instrument.
3.  **Never-Takers:** People who never take the treatment, regardless of the instrument.
4.  **Defiers:** People who do the opposite of what the instrument encourages. A key assumption for the LATE interpretation is that there are no defiers (**monotonicity**).

The IV estimator identifies the average treatment effect *only for the group of compliers*:
$$ \beta_{IV} \xrightarrow{p} E[Y(1) - Y(0) | \text{i is a complier}] = \text{LATE} $$


### Case Study: Angrist and Krueger (1991) Returns to Education


In [ ]:
try:
    ak91_df = sm.datasets.get_rdataset("ak91", "ivmodel").data
    ak91_df['log_wage'] = np.log(ak91_df['wage'])
    ak91_df['qob_is_4'] = (ak91_df['qob'] == 4).astype(int)
    display(Markdown('> **Note:** Loaded Angrist and Krueger (1991) dataset.'))
except Exception as e:
    ak91_df = None
    display(Markdown(f'> **Note:** Could not load dataset. Skipping case study. Error: {e}'))
else:
    ols_model = smf.ols('log_wage ~ school', data=ak91_df).fit()
    iv_model = IV2SLS.from_formula('log_wage ~ 1 + [school ~ qob_is_4]', data=ak91_df).fit()
    print("--- OLS Results ---"); print(ols_model.summary().tables[1])
    print("\n--- IV (2SLS) Results ---"); print(iv_model)
    display(Markdown(f'> **Note:** The OLS estimate suggests a return of {ols_model.params["school"]*100:.1f}%. The IV estimate is {iv_model.params["school"]*100:.1f}%. The LATE interpretation suggests this is the return to schooling for the \'compliers\' - those whose schooling was affected by their birth quarter.'))


> **Note:** Loaded Angrist and Krueger (1991) dataset.


### 3. Weak Instruments
A critical problem in applied IV is the presence of **weak instruments**. If the instrument is only weakly correlated with the endogenous variable, the IV estimator has poor finite-sample properties:
1.  **Bias:** The 2SLS estimator is biased towards the OLS estimator.
2.  **Non-Normal Distribution:** The sampling distribution is not well-approximated by a normal distribution, making standard t-tests unreliable.

**Detection:** The standard diagnostic is the **first-stage F-statistic**. A common rule of thumb (Staiger & Stock, 1997) is that an F-statistic **below 10** signals a potential weak instrument problem.


### Interactive: The Weak Instrument Problem


In [ ]:
def run_weak_iv_sim(instrument_strength=0.1, n_sims=1000):
    true_beta = 0.8; ols_estimates, iv_estimates, f_stats = [], [], []
    for _ in range(n_sims):
        n = 200; ability = np.random.normal(0, 1, n); instrument = np.random.normal(0, 1, n)
        education = instrument_strength * instrument + 1.2 * ability + np.random.normal(0, 1, n)
        log_wage = true_beta * education + 1.0 * ability + np.random.normal(0, 1, n)
        df = pd.DataFrame({'log_wage':log_wage, 'educ':education, 'instr':instrument})
        ols = smf.ols('log_wage ~ educ', data=df).fit()
        iv = IV2SLS.from_formula('log_wage ~ 1 + [educ ~ instr]', df).fit()
        ols_estimates.append(ols.params['educ']); iv_estimates.append(iv.params['educ'])
        f_stats.append(iv.first_stage.f.stat)

    plt.figure(figsize=(12, 6))
    sns.kdeplot(ols_estimates, label=f'OLS Estimates (Mean={np.mean(ols_estimates):.2f})', fill=True)
    sns.kdeplot(iv_estimates, label=f'IV Estimates (Mean={np.mean(iv_estimates):.2f})', fill=True)
    plt.axvline(true_beta, color='k', ls='--', label=f'True Beta = {true_beta}')
    plt.title('Distribution of OLS vs. IV Estimates'); plt.legend()
    plt.show()
    display(Markdown(f'> **Note:** With instrument strength = {instrument_strength}, the average First-Stage F-statistic is {np.mean(f_stats):.1f}.'))

widgets.interact(run_weak_iv_sim, instrument_strength=widgets.FloatSlider(min=0.0, max=0.5, step=0.02, value=0.1));


In [ ]:
# --- Two-Stage Least Squares (2SLS) from Scratch ---

class TwoStageLeastSquares:
    def __init__(self):
        self.first_stage_params = None
        self.second_stage_params = None
        self.se = None

    def fit(self, y, X, Z):
        # Add intercept
        X = sm.add_constant(X)
        Z = sm.add_constant(Z)

        # First Stage: Regress X on Z
        # X_hat = Z(Z'Z)^-1 Z'X
        self.first_stage_model = sm.OLS(X, Z).fit()
        X_hat = self.first_stage_model.predict(Z)
        self.first_stage_params = self.first_stage_model.params

        # Second Stage: Regress y on X_hat
        # beta_2sls = (X_hat' X_hat)^-1 X_hat' y
        self.second_stage_model = sm.OLS(y, X_hat).fit()
        self.second_stage_params = self.second_stage_model.params

        # Standard Errors (Must use original X, not X_hat, for residuals)
        residuals = y - self.second_stage_model.predict(X)
        sigma2 = (residuals.T @ residuals) / (len(y) - X.shape[1])
        # Var(beta) = sigma2 * (X' Pz X)^-1 where Pz is projection matrix of Z
        # Approximated by second stage covariance but corrected for sigma2
        self.se = self.second_stage_model.bse # Simplification for demonstration

        return self

    def summary(self):
        print("--- 2SLS Estimation Results ---")
        print("First Stage F-stat:", self.first_stage_model.fvalue)
        if self.first_stage_model.fvalue < 10:
            print("Warning: Weak Instruments (F < 10)")
        print("\nSecond Stage Coefficients:")
        print(self.second_stage_params)

# Generate Data with Endogeneity
np.random.seed(42)
n = 500
z = np.random.normal(0, 1, n) # Instrument
u = np.random.normal(0, 1, n) # Structural Error
v = 0.5 * u + np.random.normal(0, 1, n) # Endogeneity channel
x = 0.5 * z + v # Endogenous regressor
y = 1 + 2 * x + u

# Naive OLS
ols_biased = sm.OLS(y, sm.add_constant(x)).fit()
print(f"Biased OLS Estimate of beta (True=2.0): {ols_biased.params[1]:.4f}")

# 2SLS
iv_model = TwoStageLeastSquares()
iv_model.fit(y, pd.DataFrame(x, columns=['x']), pd.DataFrame(z, columns=['z']))
iv_model.summary()
print(f"2SLS Estimate of beta: {iv_model.second_stage_params[0]:.4f}")


### 4. The Control Function Approach
An alternative to 2SLS is the **control function** approach. Instead of purging the endogeneity from $D$, this method attempts to model the source of the endogeneity directly and include it in the regression as a control.

**Procedure:**
1.  Assume the endogeneity arises from $D = \Pi Z + v$, where $v$ is correlated with the structural error $u$. Assume $u = \rho v + \epsilon$, where $\epsilon$ is now well-behaved.
2.  **First Stage:** Run the regression of $D$ on $Z$ and obtain the residuals, $\hat{v}$.
3.  **Second Stage:** Run the original regression of $Y$ on $D$, but now include the first-stage residuals $\hat{v}$ as an additional regressor:
    $$ Y = \beta_0 + \beta_1 D + \delta \hat{v} + \epsilon $$ 

In this regression, the coefficient $\beta_1$ is a consistent estimate of the causal effect. The coefficient $\delta$ on the residual is an estimate of $\rho$, and a t-test on it is a **test for endogeneity**.


### Control Function Example and Endogeneity Test


In [ ]:
if ak91_df is not None:
    # 1. First Stage
    first_stage = smf.ols('school ~ qob_is_4', data=ak91_df).fit()
    ak91_df['resid'] = first_stage.resid

    # 2. Second Stage
    control_fn_model = smf.ols('log_wage ~ school + resid', data=ak91_df).fit()

    print(control_fn_model.summary().tables[1])


> **Note:** The coefficient on 'school' is the control function estimate of the causal effect. The coefficient on 'resid' is statistically insignificant, suggesting we cannot reject the null hypothesis that schooling is exogenous in this specification.


> **Note:** Dataset not available.


## Key Equations

These relations are collected from the derivations above as a review map. Their assumptions and derivations remain part of the result; this box is not a substitute for them.

**1. Core relation**

$$D = \pi_0 + \pi_1 Z + \pi_2 X + v$$

**2. Core relation**

$$Y = \beta_0 + \beta_1 \hat{D} + \beta_2 X + u$$

**3. Core relation**

$$\beta_{IV} \xrightarrow{p} E[Y(1) - Y(0) | \text{i is a complier}] = \text{LATE}$$

**4. Core relation**

$$Y = \beta_0 + \beta_1 D + \delta \hat{v} + \epsilon$$


### Three-Tier Practice Ladder

**1. Mechanism and assumptions (Conceptual):** Define the estimand in **05 Instrumental Variables**, list the identifying assumptions, and give a concrete data-generating process that violates one assumption while leaving the others intact.

**2. Reproduce and diagnose (Applied):** Implement or reproduce the estimator using the material on 1. The IV Estimator: Two-Stage Least Squares (2SLS), 2. Heterogeneous Effects and the LATE Framework. Report uncertainty and at least two diagnostics; then compare with an alternative specification that targets the same estimand.

**3. Robust extension (Challenge):** Run a Monte Carlo or sensitivity exercise that varies the most fragile identifying condition. Quantify bias/coverage or the range of estimates and state what evidence would change your substantive conclusion.

> Use the existing exercises above when they target the same skill; this ladder makes the intended progression explicit rather than replacing instructor-authored problems.


## 5. Exercises

1.  **IV Assumptions:** For the Angrist and Krueger (1991) study, explain in detail what the relevance and exclusion restriction assumptions imply. Why might the exclusion restriction be violated?

2.  **LATE Interpretation:** In the Angrist and Krueger study, who are the 'compliers'? Who are the 'never-takers' and 'always-takers'? Why might the LATE be different from the ATE for the returns to schooling?

3.  **Testing for Weak Instruments:** Using the `linearmodels` library on the Angrist and Krueger data, access the first-stage regression results from the `iv_model` object. What is the F-statistic for the relevance of the quarter-of-birth instrument? Based on the rule of thumb, is this instrument considered weak?

4.  **Control Function for Endogeneity Testing:** Using the synthetic data from the weak instrument simulation, implement the control function approach. Run the first stage, get the residuals, and include them in the second stage. Perform a t-test on the coefficient of the residual. Does the test correctly detect endogeneity? How does the estimated treatment effect compare to the OLS and 2SLS estimates?

5.  **Invalid Instrument:** Suppose you use a bad instrument that violates the exclusion restriction. Specifically, assume the instrument $Z$ has a direct effect on the outcome $Y$. Modify the weak instrument simulation code to include such a direct effect. How does this affect the bias of the IV estimator?


# Summary

IV is the cure for endogeneity, but valid instruments are rare.

**Key Takeaways:**
*   **LATE:** When treatment effects are heterogeneous, IV estimates the Local Average Treatment Effect—the effect on "compliers" who are shifted by the instrument.
*   **Weak Instruments:** If the correlation between $Z$ and $X$ is weak, IV estimates are biased and standard errors are misleading.
*   **Exclusion Restriction:** This assumption is untestable. It must be defended with economic theory and qualitative evidence.


## References & Further Reading

- Wooldridge, J. M. (2010). *Econometric Analysis of Cross Section and Panel Data* (2nd ed.). MIT Press.
- Angrist, J. D. & Pischke, J.-S. (2009). *Mostly Harmless Econometrics*. Princeton University Press.
- Imbens, G. W. & Rubin, D. B. (2015). *Causal Inference for Statistics, Social, and Biomedical Sciences*. Cambridge University Press.
